# dcgan-wrapper-netG-netD — worked example 3: Move both subnets to a device via the wrapper

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `dcgan-wrapper-netG-netD`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

Because `netG` and `netD` are registered submodules, a single `wrapper.to(device)` recursively moves **both** subnets' parameters and buffers. You never need to move each subnet individually — calling `.to()`, `.float()`, or `.cuda()` on the container fans out to every registered child. This is one of the practical payoffs of using an `nn.Module` wrapper instead of a plain object.

## Worked solution

**Step 1 — build the wrapper** with the standard idiom (`super().__init__()` first, assign `netG`/`netD`, no `forward`).

**Step 2 — force a known starting dtype.** We start the subnets in `float32` (the default) and assert it, so the conversion is observable.

**Step 3 — call `.to()` once on the container.** `wrapper.to(t.float64)` walks `self._modules` recursively and converts every parameter in both subnets. We use a dtype change rather than a device change because CPU-only environments can't move to CUDA, but the recursion mechanism is identical.

**Step 4 — verify both subnets followed.** We check that a parameter inside `netG` AND a parameter inside `netD` are now `float64`. Because the conversion is in-place on the registered submodules, both report the new dtype — proof the single container-level call propagated to both children.

In [ ]:
from torch import nn
import torch as t

def move_wrapper_dtype(generator, discriminator, dtype):
    class DCGAN(nn.Module):
        def __init__(self, netG, netD):
            super().__init__()
            self.netG = netG
            self.netD = netD
    wrapper = DCGAN(generator, discriminator)
    wrapper.to(dtype)          # single call, fans out to both subnets
    return wrapper

t.manual_seed(0)
gen = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 8))
disc = nn.Sequential(nn.Linear(8, 4), nn.ReLU(), nn.Linear(4, 1))
before = next(gen.parameters()).dtype
wrapper = move_wrapper_dtype(gen, disc, t.float64)
g_dtype = next(wrapper.netG.parameters()).dtype
d_dtype = next(wrapper.netD.parameters()).dtype
print('before:', before, '| netG now:', g_dtype, '| netD now:', d_dtype, '| both float64:', g_dtype == t.float64 and d_dtype == t.float64)